In [1]:
import sys
from pyspark.sql import SparkSession

In [2]:
sys.path.append(".")
spark = SparkSession.builder.appName("ETL Pipeline").config("spark.driver.memory", "2g").getOrCreate()


In [3]:
locations = {
    "SEA": ["Singapore", "Bangkok", "Jakarta", "Kuala Lumpur", "Manila", "Hanoi", "Ho Chi Minh City"],
    "NA" : ["New York", "Los Angeles", "Chicago", "Toronto", "Mexico City", "Houston", "Miami"],
    "SA" : ["São Paulo", "Buenos Aires", "Rio de Janeiro", "Lima", "Bogotá", "Santiago", "Caracas"],
    "EU" : ["London", "Paris", "Berlin", "Madrid", "Rome", "Amsterdam", "Vienna"],
    "AS" : ["Tokyo", "Beijing", "Seoul", "Mumbai", "Shanghai", "Bangkok", "Delhi"],
    "AF" : ["Cairo", "Lagos", "Johannesburg", "Nairobi", "Casablanca", "Accra", "Addis Ababa"],
    "OC" : ["Sydney", "Melbourne", "Auckland", "Brisbane", "Perth", "Fiji", "Port Moresby"], 
    "ALL": []
}

locations["ALL"] = (locations["SEA"] + locations["NA"] + locations["SA"] + locations["EU"] + locations["AS"] + locations["AF"] + locations["OC"])



In [4]:
from utils import get_weather_data,schema

In [5]:
allweather =[]
print("ETL job started")
for i in locations["ALL"]:
    data = get_weather_data(i)
    allweather.append(data)



ETL job started


In [6]:
for i in allweather:
    print(i['main']['feels_like'])

311.14
313.6
312.7
311.63
306.17
310.48
309.78
288.49
295.98
286.48
286.44
288.92
299.73
299.48
288.59
287.93
291.29
287.79
283.8
290.03
298.96
288.47
291.24
292.29
293.9
295.91
291.16
291.43
309.37
305.66
307.87
309.29
309.61
313.6
313.37
305.33
298.75
293.71
292.3
296.19
298.05
290.21
289.2
286.71
282.43
290.09
292.48
280.86
299.1


In [7]:
df = spark.createDataFrame(allweather, schema=schema)

df.show()

+--------------------+--------------------+--------+--------------------+----------+------------------+------+----------+--------------------+--------+-------+----------------+---+
|               coord|             weather|    base|                main|visibility|              wind|clouds|        dt|                 sys|timezone|     id|            name|cod|
+--------------------+--------------------+--------+--------------------+----------+------------------+------+----------+--------------------+--------+-------+----------------+---+
|  {103.8519, 1.2899}|[{802, Clouds, sc...|stations|{304.41, 311.14, ...|     10000| {3.42, 223, 3.68}|  {43}|1756626306|{SG, 1756594864, ...|   28800|1880252|       Singapore|200|
| {100.4935, 13.7525}|[{804, Clouds, ov...|stations|{306.6, 313.6, 30...|     10000| {5.41, 242, 6.83}|  {94}|1756626404|{TH, 1756595197, ...|   25200|1608132|      Nonthaburi|200|
| {106.8272, -6.1754}|[{802, Clouds, sc...|stations|{305.7, 312.7, 30...|     10000|  {3.19, 20

In [8]:
df.createOrReplaceTempView("weather")


spark.sql("""
SELECT * FROM weather    
""").show()



+--------------------+--------------------+--------+--------------------+----------+------------------+------+----------+--------------------+--------+-------+----------------+---+
|               coord|             weather|    base|                main|visibility|              wind|clouds|        dt|                 sys|timezone|     id|            name|cod|
+--------------------+--------------------+--------+--------------------+----------+------------------+------+----------+--------------------+--------+-------+----------------+---+
|  {103.8519, 1.2899}|[{802, Clouds, sc...|stations|{304.41, 311.14, ...|     10000| {3.42, 223, 3.68}|  {43}|1756626306|{SG, 1756594864, ...|   28800|1880252|       Singapore|200|
| {100.4935, 13.7525}|[{804, Clouds, ov...|stations|{306.6, 313.6, 30...|     10000| {5.41, 242, 6.83}|  {94}|1756626404|{TH, 1756595197, ...|   25200|1608132|      Nonthaburi|200|
| {106.8272, -6.1754}|[{802, Clouds, sc...|stations|{305.7, 312.7, 30...|     10000|  {3.19, 20

In [9]:
# spark.stop()

In [10]:
spark.sql("""
SELECT * FROM weather 
WHERE visibility != 10000
""").show()

+--------------------+--------------------+--------+--------------------+----------+------------------+------+----------+--------------------+--------+-------+----------------+---+
|               coord|             weather|    base|                main|visibility|              wind|clouds|        dt|                 sys|timezone|     id|            name|cod|
+--------------------+--------------------+--------+--------------------+----------+------------------+------+----------+--------------------+--------+-------+----------------+---+
| {106.7018, 10.7758}|[{804, Clouds, ov...|stations|{304.23, 309.78, ...|      4745| {4.73, 246, 6.93}| {100}|1756626405|{VN, 1756593824, ...|   25200|1566083|Ho Chi Minh City|200|
|{-58.4371, -34.6076}|[{804, Clouds, ov...|stations|{287.85, 287.93, ...|       218|{8.01, 115, 14.97}| {100}|1756626407|{AR, 1756635232, ...|  -10800|3427458|    Villa Crespo|200|
+--------------------+--------------------+--------+--------------------+----------+-----------

In [11]:
spark.sql("""
SELECT coord.lon FROM weather 
WHERE visibility != 10000
""").show()

+--------+
|     lon|
+--------+
|106.7018|
|-58.4371|
+--------+

